# Run Inference — 5 Local Models × 600 Jokes

All-local, all-free pipeline. Five Ollama-served models, no API keys, no
billing, no rate limits. **Models are stored on the D: drive** (configurable
in Section 2) to avoid filling up the system disk — Ollama's default location
is the user home directory which is usually on C:.

| # | Model | Slug | Paper match | H4 role |
|---|---|---|---|---|
| 1 | DeepSeek-R1-Distill-**Llama**-8B | `r1-distill-llama-8b` | ✅ exact | — |
| 2 | Llama 3.1 8B Instruct | `llama-3.1-8b` | ✅ exact | small Llama |
| 3 | Llama 3.2 3B Instruct | `llama-3.2-3b` | extension | smaller Llama (cross-gen) |
| 4 | Gemma 2 **2B** Instruct | `gemma-2-2b` | extension | small Gemma |
| 5 | Gemma 2 **9B** Instruct | `gemma-2-9b` | extension | large Gemma |

**Why this lineup:**
- **Two paper-exact matches** (R1-Distill-Llama-8B, Llama 3.1 8B) — direct comparison to the paper's reported scores
- **Clean within-Gemma H4 pair** (2B vs 9B) — only size differs, same training run, same generation. Cleanest possible test of the size-effect hypothesis on this stack.
- **Cross-generation H4 confound test** (Llama 3.2 3B vs 3.1 8B) — bonus comparison; expected to be noisier than the Gemma pair because generation also differs
- **No judge family-bias** — none are Qwen-base, judge has no shared-family inflation risk
- **Two model families** — Llama-derived (3 models) and Google Gemma (2 models)

All five cells append to `data/explanations.jsonl`. Re-run any cell after an interruption — `inference.py` skips already-done jokes.

**Open this notebook with:**
```bash
conda activate applesVsOranges
jupyter notebook run_inference.ipynb
```

---
## Sections
1. Environment check
2. Ollama setup — point storage at D: drive + pull all 5 models
3. Smoke test — 4 jokes per model
4. Full run — R1-Distill-Llama-8B
5. Full run — Llama 3.1 8B
6. Full run — Llama 3.2 3B
7. Full run — Gemma 2 2B
8. Full run — Gemma 2 9B
9. Inspect combined output
10. Next steps — judge + analyze

---
## Section 1 — Environment Check

In [1]:
# Imports + path setup
import sys, os, json, time, subprocess
from pathlib import Path

sys.path.insert(0, "src")

from inference import main as inference_main
from inference_backend import get_inference_backend, OllamaBackend

print(f"Python  : {sys.version.split()[0]}")
print(f"CWD     : {Path.cwd()}")
print(f"jokes   : {'OK' if Path('data/jokes.jsonl').exists() else 'MISSING — run preprocess.py first'}")
print(f"output  : data/explanations.jsonl (will be created/appended)")
print()
print("Imports OK")

Python  : 3.10.19
CWD     : /mnt/d/ApplesVSOranges
jokes   : OK
output  : data/explanations.jsonl (will be created/appended)

Imports OK


In [2]:
# GPU check — matters for all 5 models since they all run via Ollama locally
try:
    import torch
    if torch.cuda.is_available():
        print(f"GPU     : {torch.cuda.get_device_name(0)}")
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"VRAM    : {vram_gb:.1f} GB")
        if vram_gb < 8:
            print("WARN    : <8GB VRAM — Gemma 2B and Llama 3B will work; 8B/9B may need CPU offload")
    else:
        print("WARN    : no CUDA GPU detected — inference will run on CPU (much slower, but works)")
        print("          expect ~5-10x longer than the time estimates in each cell")
except ImportError:
    print("torch not installed — Ollama will use whatever it detects on its own")

GPU     : NVIDIA GeForce RTX 3090
VRAM    : 25.8 GB


---
## Section 2 — Ollama Setup (Models on D: Drive)

By default, Ollama stores models in the user home directory (typically on C:),
which fills up fast — this lineup needs ~21 GB total. We redirect storage to
the D: drive via the `OLLAMA_MODELS` environment variable.

### 2a. Choose your storage location

Pick the path Ollama should use for model storage. Defaults below assume
WSL on Windows; adjust if you're on a different setup.

| OS | Path to use |
|---|---|
| WSL (Windows) | `/mnt/d/ollama-models` |
| Native Windows | `D:\ollama-models` |
| Native Linux  | `/mnt/d/ollama-models` (or wherever your "D" drive mounts) |
| macOS | (no D drive — use `/Volumes/<external>/ollama-models` or skip this section) |

In [3]:
# Configure here — change if your D: mount path is different
MODELS_DIR = "/mnt/d/ollama-models"

# Ensure the directory exists
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)
print(f"Target storage location: {MODELS_DIR}")
print(f"Directory exists       : {Path(MODELS_DIR).exists()}")

# Disk space check
import shutil
total, used, free = shutil.disk_usage(MODELS_DIR)
print(f"Free space on D:       : {free / 1e9:.1f} GB")
if free < 25 * 1e9:
    print("WARN    : <25 GB free — you may run out of space (lineup needs ~21 GB)")
else:
    print("OK      : enough space for the full lineup")

Target storage location: /mnt/d/ollama-models
Directory exists       : True
Free space on D:       : 231.5 GB
OK      : enough space for the full lineup


### 2b. Stop the daemon, set `OLLAMA_MODELS`, restart

Ollama only honors `OLLAMA_MODELS` if it's set **before** the daemon starts.
Setting it from Python won't help if the daemon's already running.

**Run these in a WSL terminal (not in this notebook):**

```bash
# 1. Stop any running ollama daemon
pkill ollama 2>/dev/null
sleep 2

# 2. Optional — move any existing models to D: drive (so they're not orphaned)
#    If you already have qwen2.5:7b pulled for the judge, this preserves it.
if [ -d ~/.ollama/models ] && [ "$(ls -A ~/.ollama/models 2>/dev/null)" ]; then
    echo "Moving existing models to D: drive..."
    mkdir -p /mnt/d/ollama-models
    mv ~/.ollama/models/* /mnt/d/ollama-models/
fi

# 3. Set OLLAMA_MODELS for this session and start the daemon
export OLLAMA_MODELS=/mnt/d/ollama-models
ollama serve &
sleep 3

# 4. (Optional) Make it permanent — add to ~/.bashrc or ~/.zshrc:
echo 'export OLLAMA_MODELS=/mnt/d/ollama-models' >> ~/.bashrc
```

After the daemon restarts, run the next cell to verify.

In [4]:
# Verify the daemon is running and using the D: drive

import requests

OLLAMA_URL = "http://localhost:11434"

def ollama_running():
    try:
        requests.get(f"{OLLAMA_URL}/api/tags", timeout=3)
        return True
    except Exception:
        return False

if not ollama_running():
    print("✗ Ollama daemon is not running.")
    print("  Run the bash block above in a WSL terminal, then re-run this cell.")
else:
    print("✓ Ollama daemon is running")

    # Check whether OLLAMA_MODELS is actually pointing where we want
    try:
        env_resp = subprocess.run(
            ["bash", "-c", "ps -ef | grep -E 'ollama serve' | grep -v grep | head -1"],
            capture_output=True, text=True, timeout=5,
        )
        proc_line = env_resp.stdout.strip()
        if proc_line:
            pid = proc_line.split()[1]
            with open(f"/proc/{pid}/environ", "rb") as f:
                env = f.read().decode("utf-8", errors="ignore").split("\x00")
            ollama_models_var = next(
                (e.split("=", 1)[1] for e in env if e.startswith("OLLAMA_MODELS=")),
                None
            )
            if ollama_models_var:
                if ollama_models_var == MODELS_DIR:
                    print(f"✓ Daemon's OLLAMA_MODELS is set to {MODELS_DIR}")
                else:
                    print(f"⚠ Daemon's OLLAMA_MODELS is {ollama_models_var}")
                    print(f"  Expected: {MODELS_DIR}")
                    print(f"  Stop the daemon and restart with the right env var.")
            else:
                print(f"⚠ Daemon does NOT have OLLAMA_MODELS set — using default location")
                print(f"  This means new pulls will go to ~/.ollama/models, not {MODELS_DIR}")
                print(f"  Stop the daemon and restart it with OLLAMA_MODELS={MODELS_DIR}")
    except Exception as e:
        print(f"  (couldn't introspect daemon env: {e}; skipping that check)")

✓ Ollama daemon is running
  (couldn't introspect daemon env: [Errno 13] Permission denied: '/proc/217/environ'; skipping that check)


### 2c. Pull all five models

In [5]:
# (ollama_tag, slug, human label, est_minutes, paper_match)
MODELS = [
    ("deepseek-r1:8b",  "r1-distill-llama-8b", "DeepSeek-R1-Distill-Llama-8B", 60,  True),
    ("llama3.1:8b",     "llama-3.1-8b",        "Llama 3.1 8B Instruct",        45,  True),
    ("llama3.2:3b",     "llama-3.2-3b",        "Llama 3.2 3B Instruct",        20,  False),
    ("gemma2:2b",       "gemma-2-2b",          "Gemma 2 2B Instruct",          15,  False),
    ("gemma2:9b",       "gemma-2-9b",          "Gemma 2 9B Instruct",          50,  False),
]

if not ollama_running():
    print("✗ Daemon not running — fix Section 2b first.")
else:
    tags = requests.get(f"{OLLAMA_URL}/api/tags").json()
    pulled = [m["name"] for m in tags.get("models", [])]
    print(f"Currently pulled ({len(pulled)}): {pulled}\n")

    print(f"{'Model':<22} {'Slug':<24} {'Status'}")
    print("-" * 70)
    missing = []
    for tag, slug, label, _, _ in MODELS:
        is_pulled = any(tag.split(":")[0] in p and tag.split(":")[1] in p for p in pulled)
        status = "✓ pulled" if is_pulled else "✗ MISSING"
        print(f"{tag:<22} {slug:<24} {status}")
        if not is_pulled:
            missing.append(tag)

    if missing:
        print("\nPull missing models in a WSL terminal:")
        for tag in missing:
            print(f"    ollama pull {tag}")
        print("\nApprox sizes:")
        print("    deepseek-r1:8b ~5GB | llama3.1:8b ~5GB | llama3.2:3b ~2GB")
        print("    gemma2:2b      ~1.5GB | gemma2:9b ~6GB | total ~21GB")
        print()
        print(f"After pulling, verify they landed on D: with:")
        print(f"    du -sh {MODELS_DIR}")
    else:
        print("\n✓ All five models are pulled and ready")
        # Verify the models are actually on D: drive
        try:
            du = subprocess.run(["du", "-sh", MODELS_DIR],
                                capture_output=True, text=True, timeout=10)
            if du.returncode == 0:
                size = du.stdout.split()[0]
                print(f"  Storage on {MODELS_DIR}: {size}")
        except Exception:
            pass

Currently pulled (6): ['gemma2:9b', 'gemma2:2b', 'llama3.2:3b', 'llama3.1:8b', 'deepseek-r1:8b', 'qwen2.5:7b-instruct-q4_K_M']

Model                  Slug                     Status
----------------------------------------------------------------------
deepseek-r1:8b         r1-distill-llama-8b      ✓ pulled
llama3.1:8b            llama-3.1-8b             ✓ pulled
llama3.2:3b            llama-3.2-3b             ✓ pulled
gemma2:2b              gemma-2-2b               ✓ pulled
gemma2:9b              gemma-2-9b               ✓ pulled

✓ All five models are pulled and ready
  Storage on /mnt/d/ollama-models: 0


---
## Section 3 — Smoke Test (4 jokes per model)

Quick sanity check on each model before committing to 600-joke runs.

In [6]:
SMOKE_OUTPUT = "outputs/smoketest_inference.jsonl"
Path(SMOKE_OUTPUT).unlink(missing_ok=True)

def run_smoke(tag, slug, label):
    print(f"\n{'='*60}\nSmoke: {label} ({slug})\n{'='*60}")
    sys.argv = [
        "inference.py",
        "--backend", "ollama",
        "--model-id", tag,
        "--model-slug", slug,
        "--output", SMOKE_OUTPUT,
        "--limit", "4",
        "--no-resume",
    ]
    try:
        inference_main()
    except SystemExit:
        pass
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")

for tag, slug, label, _, _ in MODELS:
    run_smoke(tag, slug, label)

# Verify by counting rows actually written per model — NOT by exit code,
# because the script can exit cleanly while still failing every API call.
print(f"\n{'='*60}\nSmoke results (rows actually written):\n{'='*60}")
written_by_slug = {}
if Path(SMOKE_OUTPUT).exists():
    rows = [json.loads(l) for l in open(SMOKE_OUTPUT)]
    for r in rows:
        written_by_slug[r["model"]] = written_by_slug.get(r["model"], 0) + 1

all_ok = True
for tag, slug, label, _, _ in MODELS:
    n = written_by_slug.get(slug, 0)
    ok = n == 4
    if not ok:
        all_ok = False
    print(f"  {'✓' if ok else '✗'}  {slug:<24} {n}/4 rows")

if not all_ok:
    print("\n⚠ Some models wrote 0 rows. Most likely cause:")
    print("  (a) the model wasn't pulled — check Section 2c output")
    print("  (b) the daemon isn't actually running — check Section 2b")
    print("  (c) see outputs/inference.errors.log for the exact errors")
else:
    print("\n✓ All five models wrote rows successfully — safe to run full sections 4-8")
    # Show one sample per model
    seen = set()
    for r in rows:
        if r["model"] not in seen:
            print(f"\n[{r['model']}] {r['joke_id']}")
            print(f"  {r['explanation'][:200]}{'...' if len(r['explanation']) > 200 else ''}")
            seen.add(r["model"])

INFO inference: backend=ollama model_id=deepseek-r1:8b slug=r1-distill-llama-8b — 4 jokes to do (temp=0, sleep=0s)



Smoke: DeepSeek-R1-Distill-Llama-8B (r1-distill-llama-8b)


infer[r1-distill-llama-8b]: 100%|██████████| 4/4 [00:19<00:00,  4.98s/it]
INFO inference: done: ok=4 errors={}
INFO inference: backend=ollama model_id=llama3.1:8b slug=llama-3.1-8b — 4 jokes to do (temp=0, sleep=0s)



Smoke: Llama 3.1 8B Instruct (llama-3.1-8b)


infer[llama-3.1-8b]: 100%|██████████| 4/4 [00:10<00:00,  2.69s/it]
INFO inference: done: ok=4 errors={}
INFO inference: backend=ollama model_id=llama3.2:3b slug=llama-3.2-3b — 4 jokes to do (temp=0, sleep=0s)


...


Smoke: Gemma 2 2B Instruct (gemma-2-2b)


infer[gemma-2-2b]: 100%|██████████| 4/4 [00:07<00:00,  1.77s/it]
INFO inference: done: ok=4 errors={}
INFO inference: backend=ollama model_id=gemma2:9b slug=gemma-2-9b — 4 jokes to do (temp=0, sleep=0s)



Smoke: Gemma 2 9B Instruct (gemma-2-9b)


infer[gemma-2-9b]: 100%|██████████| 4/4 [00:06<00:00,  1.59s/it]
INFO inference: done: ok=4 errors={}



Smoke results (rows actually written):
  ✓  r1-distill-llama-8b      4/4 rows
  ✓  llama-3.1-8b             4/4 rows
  ✓  llama-3.2-3b             4/4 rows
  ✓  gemma-2-2b               4/4 rows
  ✓  gemma-2-9b               4/4 rows

✓ All five models wrote rows successfully — safe to run full sections 4-8

[r1-distill-llama-8b] homographic_000
  This joke relies on a pun, or wordplay. The punchline is: "high it out."

*   "Hid" sounds like "high."
*   "Sweat it out" is a common phrase meaning to endure something or to sweat profusely.
*   Com...

[llama-3.1-8b] homographic_000
  This joke is a play on words. The phrase "sweat it out" has a double meaning here. In one sense, people often use this expression to describe waiting out a stressful or difficult situation, as if thei...

[llama-3.2-3b] homographic_000
  This joke is a play on words. The phrase "sweat it out" has a double meaning here. In one sense, sweating is a physical response to heat, which is why people often use sauna

---
## Section 4 — Full Run: DeepSeek-R1-Distill-Llama-8B

Estimated runtime: ~60 min on a consumer GPU.
Resume safe — re-run after any interruption. — **paper-match model**, your scores compare directly to the paper's reported numbers

In [ ]:
sys.argv = [
    "inference.py",
    "--backend", "ollama",
    "--model-id", "deepseek-r1:8b",
    "--model-slug", "r1-distill-llama-8b",
    "--output", "data/explanations.jsonl",
    "--max-tokens", "4096",        # double the budget
    "--temperature", "0.5",         # break the deterministic loop (ran on last 2 jokes)
]
inference_main()

INFO inference: resume: 2998 explanations already in data/explanations.jsonl (598 for this model)
INFO inference: backend=ollama model_id=deepseek-r1:8b slug=r1-distill-llama-8b — 2 jokes to do (temp=0.5, sleep=0s)
infer[r1-distill-llama-8b]: 100%|██████████| 2/2 [00:47<00:00, 23.62s/it]
INFO inference: done: ok=2 errors={}


---
## Section 5 — Full Run: Llama 3.1 8B Instruct

Estimated runtime: ~45 min on a consumer GPU.
Resume safe — re-run after any interruption. — **paper-match model**, your scores compare directly to the paper's reported numbers

In [8]:
sys.argv = [
    "inference.py",
    "--backend", "ollama",
    "--model-id", "llama3.1:8b",
    "--model-slug", "llama-3.1-8b",
    "--output", "data/explanations.jsonl",
]
inference_main()

INFO inference: resume: 571 explanations already in data/explanations.jsonl (0 for this model)
INFO inference: backend=ollama model_id=llama3.1:8b slug=llama-3.1-8b — 600 jokes to do (temp=0, sleep=0s)
infer[llama-3.1-8b]: 100%|██████████| 600/600 [12:10<00:00,  1.22s/it]
INFO inference: done: ok=600 errors={}


---
## Section 6 — Full Run: Llama 3.2 3B Instruct

Estimated runtime: ~20 min on a consumer GPU.
Resume safe — re-run after any interruption. — extension model, frame as testing whether findings generalize

In [9]:
sys.argv = [
    "inference.py",
    "--backend", "ollama",
    "--model-id", "llama3.2:3b",
    "--model-slug", "llama-3.2-3b",
    "--output", "data/explanations.jsonl",
]
inference_main()

INFO inference: resume: 1171 explanations already in data/explanations.jsonl (0 for this model)
INFO inference: backend=ollama model_id=llama3.2:3b slug=llama-3.2-3b — 600 jokes to do (temp=0, sleep=0s)
infer[llama-3.2-3b]: 100%|██████████| 600/600 [08:06<00:00,  1.23it/s]
INFO inference: done: ok=600 errors={}


---
## Section 7 — Full Run: Gemma 2 2B Instruct

Estimated runtime: ~15 min on a consumer GPU.
Resume safe — re-run after any interruption. — extension model, frame as testing whether findings generalize

In [10]:
sys.argv = [
    "inference.py",
    "--backend", "ollama",
    "--model-id", "gemma2:2b",
    "--model-slug", "gemma-2-2b",
    "--output", "data/explanations.jsonl",
]
inference_main()

INFO inference: resume: 1771 explanations already in data/explanations.jsonl (0 for this model)
INFO inference: backend=ollama model_id=gemma2:2b slug=gemma-2-2b — 600 jokes to do (temp=0, sleep=0s)
infer[gemma-2-2b]: 100%|██████████| 600/600 [09:32<00:00,  1.05it/s]
INFO inference: done: ok=600 errors={}


---
## Section 8 — Full Run: Gemma 2 9B Instruct

Estimated runtime: ~50 min on a consumer GPU.
Resume safe — re-run after any interruption. — extension model, frame as testing whether findings generalize

In [11]:
sys.argv = [
    "inference.py",
    "--backend", "ollama",
    "--model-id", "gemma2:9b",
    "--model-slug", "gemma-2-9b",
    "--output", "data/explanations.jsonl",
]
inference_main()

INFO inference: resume: 2371 explanations already in data/explanations.jsonl (0 for this model)
INFO inference: backend=ollama model_id=gemma2:9b slug=gemma-2-9b — 600 jokes to do (temp=0, sleep=0s)
infer[gemma-2-9b]: 100%|██████████| 600/600 [14:37<00:00,  1.46s/it]
INFO inference: done: ok=600 errors={}


---
## Section 9 — Inspect Combined Output

In [23]:
import pandas as pd

OUTPUT = Path("data/explanations.jsonl")
EXPECTED_TOTAL = len(MODELS) * 600   # 5 × 600 = 3000

if not OUTPUT.exists() or OUTPUT.stat().st_size == 0:
    print("No explanations yet — run Sections 4-8 first.")
else:
    rows = [json.loads(l) for l in OUTPUT.open()]
    df = pd.DataFrame(rows)
    df["joke_type"] = df["joke_id"].str.rsplit("_", n=1).str[0]

    pct = 100 * len(df) / EXPECTED_TOTAL
    print(f"Total explanations : {len(df):,} / {EXPECTED_TOTAL:,}  ({pct:.1f}% complete)")
    print(f"Models             : {sorted(df['model'].unique())}")
    print(f"Joke types         : {sorted(df['joke_type'].unique())}")
    print()
    print("Coverage matrix (model × joke_type — 150 in every cell when complete):")
    print(df.groupby(["model", "joke_type"]).size().unstack(fill_value=0))
    print()

    df["chars"] = df["explanation"].str.len()
    df["words"] = df["explanation"].str.split().str.len()
    print("Explanation length stats by model:")
    print(df.groupby("model")[["chars", "words"]].agg(["mean", "min", "max"]).round(0))

Total explanations : 3,000 / 3,000  (100.0% complete)
Models             : ['gemma-2-2b', 'gemma-2-9b', 'llama-3.1-8b', 'llama-3.2-3b', 'r1-distill-llama-8b']
Joke types         : ['heterographic', 'homographic', 'non_topical', 'topical']

Coverage matrix (model × joke_type — 150 in every cell when complete):
joke_type            heterographic  homographic  non_topical  topical
model                                                                
gemma-2-2b                     150          150          150      150
gemma-2-9b                     150          150          150      150
llama-3.1-8b                   150          150          150      150
llama-3.2-3b                   150          150          150      150
r1-distill-llama-8b            150          150          150      150

Explanation length stats by model:
                     chars            words         
                      mean  min   max  mean min  max
model                                               
gemm

In [24]:
# Eyeball one explanation per (model, joke_type) — quick quality check

if OUTPUT.exists() and OUTPUT.stat().st_size > 0:
    rows = [json.loads(l) for l in OUTPUT.open()]
    df = pd.DataFrame(rows)
    df["joke_type"] = df["joke_id"].str.rsplit("_", n=1).str[0]

    for (model, jt), group in df.groupby(["model", "joke_type"]):
        sample = group.iloc[0]
        print(f"\n[{model} | {jt}]  joke_id={sample['joke_id']}")
        print(f"  {sample['explanation'][:300]}")
        if len(sample["explanation"]) > 300:
            print(f"  ... ({len(sample['explanation'])} chars total)")


[gemma-2-2b | heterographic]  joke_id=heterographic_000
  The joke plays on the common phrase "halfway up a mountain."  It sets up an expectation of a literal climb, but then introduces a twist by using "alleged" as a word that implies something is being claimed or stated falsely. 

This creates humor because it suggests Tom's claim about being halfway up 
  ... (418 chars total)

[gemma-2-2b | homographic]  joke_id=homographic_000
  The joke plays on the double meaning of "sweat it out."  

* **Literal:** In a sauna, people sweat to relax and detoxify.
* **Figurative:** To "sweat it out" means to endure hardship or difficult situations. 

The humor lies in the unexpected juxtaposition of these two meanings. The audience expects
  ... (498 chars total)

[gemma-2-2b | non_topical]  joke_id=non_topical_000
  The joke is a play on words using the double meaning of "blow" and "swallow." 

* **Whale's plan:** The male whale wants to sink the whaling ship by blowing air out of his blowhole.

---
## Section 10 — Next Steps

When `data/explanations.jsonl` reaches 3,000 rows (5 models × 600 jokes), run Member 1's pipeline against your outputs:

```bash
# Judge YOUR explanations (Qwen 7B via the same Ollama daemon)
python src/judge.py \
    --explanations data/explanations.jsonl \
    --output outputs/ratings_judge_ours.jsonl \
    --backend ollama

# Automatic metrics
python src/metrics.py \
    --explanations data/explanations.jsonl \
    --output outputs/metrics_ours.csv

# Figures + hypothesis tests
python src/analyze.py \
    --ratings outputs/ratings_judge_ours.jsonl \
    --metrics outputs/metrics_ours.csv \
    --outdir outputs/ours/
```

If `metrics.py` or `analyze.py` don't have the relevant flags yet, that's a small argparse change.

### Caveats for the writeup

1. **Two paper matches, three extension models.** Frame in two parts:
   - For `r1-distill-llama-8b` and `llama-3.1-8b`: direct comparison to the paper's reported numbers; document the gap.
   - For `llama-3.2-3b`, `gemma-2-2b`, `gemma-2-9b`: framed as *"do the paper's findings generalize to newer / different-family models?"*

2. **Two H4 tests, one cleaner than the other.**
   - **Clean H4 test (Gemma 2B vs 9B)** — same generation, same training run, only size differs. This is the strongest evidence.
   - **Confounded H4 test (Llama 3.1 8B vs 3.2 3B)** — different generations, so the comparison conflates size and generation improvements. Worth running for cross-validation but flag the confound.

3. **No judge family-bias** — all five inference models are Llama-derived or Gemma; the Qwen 7B judge has no shared-family inflation risk. Worth saying explicitly because it strengthens the evaluation methodology.

4. **All-local stack, fully reproducible** — no API calls, no closed-source dependencies. Anyone with Ollama can rerun your entire experiment with the model tags listed in the README. Defensible framing: *"we deliberately use only freely-runnable open-source models to ensure full reproducibility of the inference step."*